# Cine-Cutie 复现与模型工程证据

本 Notebook 不需要 API Key，可验证代码、测试、H3 工作流哈希和量化配置。GPU 生成必须连接实际 ComfyUI 后另行执行，未实测数据不会被伪造。

In [ ]:
from pathlib import Path
import json, subprocess, hashlib, platform

root = Path.cwd()
if not (root / 'package.json').exists():
    root = root.parent
print({'root': str(root), 'python': platform.python_version(), 'platform': platform.platform()})

## 1. 构建与测试
执行仓库统一验收入口；失败时保留完整 stdout/stderr 作为复现证据。

In [ ]:
verify = subprocess.run(['npm', 'run', 'verify'], cwd=root, text=True, capture_output=True, shell=platform.system() == 'Windows')
print(verify.stdout[-4000:])
if verify.returncode:
    print(verify.stderr[-4000:])
verify.check_returncode()

## 2. 生成工作流与量化证据
固定 seed=42，结果写入 `reports/benchmark.json`。

In [ ]:
bench = subprocess.run(['npm', 'run', 'benchmark'], cwd=root, text=True, capture_output=True, shell=platform.system() == 'Windows')
print(bench.stdout)
bench.check_returncode()
report = json.loads((root / 'reports' / 'benchmark.json').read_text(encoding='utf-8'))
report

## 3. 独立校验工作流 SHA-256

In [ ]:
actual = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((root / 'server' / 'workflows').glob('*.json'))}
recorded = {item['name']: item['sha256'] for item in report['workflows']}
assert actual == recorded
actual

## 4. GPU 实测待填项

连接 DGX 后，将每档 profile 的 prompt ID、排队时间、推理时间、峰值显存、输出哈希和 QC 分数附加到报告。只有来自 `/queue`、`/history/<prompt_id>` 和监控接口的实际值才能标记为 measured。